# DiffusionGemma Sampling

Example on how to load a DiffusionGemma model and run inference on it.

Unlike autoregressive (AR) models that generate tokens one at a time left-to-right, DiffusionGemma generates text by iteratively denoising a block ("canvas") of tokens in parallel.

The diffusion `ChatSampler` provides the same interface as the AR `gm.text.ChatSampler`, so you can use it as a drop-in replacement.

In [1]:
!pip install -q gemma

In [2]:
# Common imports
import os
import jax

# Gemma imports
  from gemma import gm
  from gemma import diffusion

## Load model and checkpoint

Load the DiffusionGemma model and its parameters.

In [ ]:
# Initialize the Diffusion Gemma model
model = diffusion.DiffusionGemma_A26B_A4B()

# Load pretrained parameters
params = gm.ckpts.load_params(diffusion.CheckpointPath.DIFFUSIONGEMMA_A26B_A4B_IT, restore_concurrent_gb=16)
print("Checkpoint loaded.")

## Sampling the model

The easiest way to use Diffusion Gemma is through `diffusion.ChatSampler`.

- **`canvas_length`**: Size of the token block denoised in parallel (default: 256).
- **`max_denoising_steps`**: Refinement steps per canvas (default: 48).


In [ ]:
# Initialize the tokenizer
tokenizer = gm.text.Gemma4Tokenizer()

# Create the ChatSampler with diffusion-specific parameters
sampler = diffusion.ChatSampler(
    model=model,
    params=params,
    tokenizer=tokenizer,
    canvas_length=256,
    max_denoising_steps=48,
    # Settings for example colab only
    pad_length=1024,  # Pad prompt to 1024 tokens
    max_out_length=3072,  # Allocate the rest of the cache for the output
)

In [ ]:
### Compile the model

DiffusionGemma runs on JAX, which traces and compiles the model on the first call. After this one-time step (~1-2 minutes), all subsequent calls are fast.

In [ ]:
print("Compiling model (one-time)...")
_ = sampler.chat("warmup")
print("Done! Model is ready.")

In [ ]:
prompt = "What are diffusion LLMs?"
print(f"Prompt: {prompt}")
print("Generating response...")
response = sampler.chat(prompt)
print(f"Response:\n{response}")